# 🥭 Mangosteen Ripeness Classification — Separable CNN 64 (Edge AI for ESP32-S3)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Supot77/Mini-Mangosteen-Detect/blob/main/training/Mangosteen_Training_SeparableCNN64.ipynb)

สมุดโน้ต (Jupyter Notebook) สำหรับฝึกสอนโมเดล **Separable CNN 64** ซึ่งเป็นโมเดลเวอร์ชันล่าสุดของโปรเจกต์ตรวจจับความสุกผลมังคุดสำหรับบอร์ดไมโครคอนโทรลเลอร์ **LilyGO T-SIMCAM (ESP32-S3)** พร้อมกล้อง **OV2640**

---

### 📋 สรุปคุณสมบัติโมเดลล่าสุด (Model Contract & Specifications):
- **สถาปัตยกรรมโมเดล:** Depthwise-Separable CNN (`Separable CNN 64`)
- **ขนาดพารามิเตอร์:** **61,059 parameters** (ผ่านเกณฑ์เข้มงวดไม่เกิน 100,000 parameters)
- **Input Dimension:** $96 \times 96 \times 3$ RGB (Normalized $[-1.0, 1.0]$)
- **Output Classes (3 คลาส):**
  0. `overripe` (มังคุดสุกเกิน / เนื้อเสีย)
  1. `ripe` (มังคุดสุกพร้อมรับประทาน)
  2. `unripe` (มังคุดดิบ)
- **ประสิทธิภาพที่ผ่านการทดสอบ:**
  - Validation Accuracy: **91.18% (31/34)**
  - Float32 Test Accuracy: **94.74% (36/38)** (Balanced Accuracy **96.30%**, Macro F1 **0.905**)
  - Full INT8 Quantized Test Accuracy: **94.74% (36/38)** (Zero quantization loss!)
  - Deployment-Matched Crop Check: **100.00% (38/38)**
- **ขนาดโมเดล TFLite (Full INT8):** **~100.5 KiB** (102,912 bytes) เหมาะอย่างยิ่งสำหรับหน่วยความจำบน ESP32-S3
- **Hardware Target:** LilyGO T-SIMCAM (ESP32-S3, 16MB Flash, 8MB PSRAM, OV2640 Camera)


## 1. ติดตั้งแพ็กเกจและเตรียมสภาพแวดล้อม (Environment Setup)
เซลล์นี้จะตรวจสอบ GPU runtime และโคลน GitHub Repository หรือดึงข้อมูลโปรเจกต์เข้ามาทำงานใน Colab


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# ตรวจสอบสภาพแวดล้อม GPU
print("Python version:", sys.version)
try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print("✅ GPU detected:", gpus)
    else:
        print("ℹ️ Running on CPU (สามารถใช้งานได้ปกติ โมเดลขนาดกะทัดรัดเทรนได้เร็ว)")
except ImportError:
    print("TensorFlow not installed. Installing required packages...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow", "scikit-learn", "matplotlib", "pillow", "seaborn"], check=True)
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)

# โคลนโปรเจกต์จาก GitHub หรือตั้งค่า Path
REPO_URL = "https://github.com/Supot77/Mini-Mangosteen-Detect.git"
REPO_DIR = Path("/content/Mini-Mangosteen-Detect")

if Path("/content").exists():
    # อยู่บน Google Colab
    if not REPO_DIR.exists():
        print("📥 กำลังโคลนโปรเจกต์จาก GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    PROJECT_ROOT = REPO_DIR
else:
    # รันบนเครื่อง Local หรือ Jupyter ธรรมดา
    PROJECT_ROOT = Path(".").resolve()
    if not (PROJECT_ROOT / "current_model").exists() and (PROJECT_ROOT.parent / "current_model").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

print("📁 PROJECT_ROOT:", PROJECT_ROOT)
assert PROJECT_ROOT.exists(), f"ไม่พบโฟลเดอร์โปรเจกต์: {PROJECT_ROOT}"


## 2. ตรวจสอบและโหลดชุดข้อมูล (Dataset Loading & Inspection)
โมเดลนี้ใช้ชุดข้อมูลที่เป็นมาตรฐานของโปรเจกต์:
- **`current_model/dataset_deployment_matched_v4`** (ชุดข้อมูลทางการที่ใช้ใน run ล่าสุด: Train 214, Val 34, Test 38)
- และรองรับการโหลดจาก `training/Mangosteen_EdgeAI/01_data/dataset` ด้วย


In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

CLASSES = ["overripe", "ripe", "unripe"]
NUM_CLASSES = len(CLASSES)
IMG_SIZE = (96, 96)
INPUT_SHAPE = (96, 96, 3)

# ตรวจสอบตำแหน่ง Dataset ที่มีอยู่ในโปรเจกต์
candidate_dataset_dirs = [
    PROJECT_ROOT / "current_model" / "dataset_deployment_matched_v4",
    PROJECT_ROOT / "training" / "Mangosteen_EdgeAI" / "01_data" / "dataset",
    PROJECT_ROOT / "01_data" / "dataset",
]

DATASET_ROOT = next((p for p in candidate_dataset_dirs if p.exists() and (p / "train").exists()), None)
print("📂 ตำแหน่ง Dataset ที่เลือกใช้:", DATASET_ROOT)
assert DATASET_ROOT is not None, "❌ ไม่พบโฟลเดอร์ Dataset กรุณาตรวจสอบโครงสร้างโฟลเดอร์"

def load_split_data(dataset_dir: Path, split: str):
    images, labels, filenames = [], [], []
    split_dir = dataset_dir / split
    for idx, class_name in enumerate(CLASSES):
        class_folder = split_dir / class_name
        if not class_folder.exists():
            continue
        for file_path in sorted(class_folder.glob("*")):
            if file_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                with Image.open(file_path) as img:
                    # ปรับเป็น 96x96 และ RGB
                    img_resized = img.convert("RGB").resize(IMG_SIZE)
                    images.append(np.asarray(img_resized, dtype=np.float32))
                    labels.append(idx)
                    filenames.append(file_path.name)
    return np.asarray(images, dtype=np.float32), np.asarray(labels, dtype=np.int32), filenames

x_train, y_train, train_files = load_split_data(DATASET_ROOT, "train")
x_val, y_val, val_files = load_split_data(DATASET_ROOT, "val")
x_test, y_test, test_files = load_split_data(DATASET_ROOT, "test")

print(f"📊 สรุปจำนวนรูปภาพในแต่ละชุด:")
print(f"  - Train Set     : {len(x_train)} ภาพ (Shape: {x_train.shape})")
print(f"  - Validation Set: {len(x_val)} ภาพ (Shape: {x_val.shape})")
print(f"  - Test Set      : {len(x_test)} ภาพ (Shape: {x_test.shape})")

# แสดงแจกแจงจำนวนแต่ละคลาส
print("\n📋 จำนวนภาพแยกตามคลาส:")
for idx, cls in enumerate(CLASSES):
    train_c = np.sum(y_train == idx)
    val_c = np.sum(y_val == idx)
    test_c = np.sum(y_test == idx)
    print(f"  - {cls:<10}: Train={train_c:>3} | Val={val_c:>3} | Test={test_c:>3} | รวม={train_c + val_c + test_c:>3}")


### แสดงภาพตัวอย่างจากชุดข้อมูลแต่ละคลาส (Dataset Preview)


In [ ]:
# แสดงภาพตัวอย่าง 3 คลาส
plt.figure(figsize=(10, 4))
for idx, class_name in enumerate(CLASSES):
    class_indices = np.where(y_train == idx)[0]
    if len(class_indices) > 0:
        sample_idx = class_indices[0]
        plt.subplot(1, 3, idx + 1)
        plt.imshow(x_train[sample_idx].astype(np.uint8))
        plt.title(f"{class_name} (Class {idx})\n96x96 RGB")
        plt.axis("off")
plt.tight_layout()
plt.show()


## 3. สร้างโครงสร้างสถาปัตยกรรมโมเดล Separable CNN 64
โมเดลถูกออกแบบขึ้นเป็นพิเศษเพื่อรองรับ **Edge AI บน ESP32-S3**:
1. **ควบคุม Parameters:** ให้ต่ำกว่า 100,000 parameters อย่างเข้มงวด (โมเดลนี้มี **61,059 parameters**)
2. **Depthwise-Separable Convolutions:** แยก spatial filtering และ depthwise feature combination ออกจากกัน ช่วยลดการคำนวณและหน่วยความจำได้มากกว่า 8-9 เท่าเมื่อเทียบกับ Standard Convolution
3. **Rescaling Layer ในตัว:** แปลงค่าสีจากช่วง $[0, 255]$ ไปเป็น $[-1.0, 1.0]$ อัตโนมัติ
4. **Batch Normalization + ReLU6:** รองรับ Full INT8 Quantization ได้อย่างเสถียร ไม่เกิด Overflow บน DSP/SIMD ของ ESP32-S3


In [ ]:
VARIANT_BLOCKS = [
    (32, 2),  # SepConv 32 filters, stride 2
    (32, 1),  # SepConv 32 filters, stride 1
    (64, 2),  # SepConv 64 filters, stride 2
    (64, 1),  # SepConv 64 filters, stride 1
    (96, 2),  # SepConv 96 filters, stride 2
    (96, 1),  # SepConv 96 filters, stride 1
    (128, 2), # SepConv 128 filters, stride 2
    (128, 1), # SepConv 128 filters, stride 1
]

def build_separable_cnn_64(input_shape=(96, 96, 3), num_classes=3) -> tf.keras.Model:
    inputs = tf.keras.layers.Input(shape=input_shape, name="input_layer")
    
    # 1. Normalization [-1.0, 1.0]
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="input_rescaling")(inputs)
    
    # 2. Stem Convolution
    first_filters = VARIANT_BLOCKS[0][0]
    x = tf.keras.layers.Conv2D(first_filters, 3, strides=2, padding="same", use_bias=False, name="stem_conv")(x)
    x = tf.keras.layers.BatchNormalization(name="stem_bn")(x)
    x = tf.keras.layers.ReLU(6.0, name="stem_relu")(x)
    
    # 3. Depthwise-Separable Blocks
    for index, (filters, stride) in enumerate(VARIANT_BLOCKS, start=1):
        x = tf.keras.layers.SeparableConv2D(
            filters, 3, strides=stride, padding="same", use_bias=False,
            name=f"sepconv_{index:02d}",
        )(x)
        x = tf.keras.layers.BatchNormalization(name=f"bn_{index:02d}")(x)
        x = tf.keras.layers.ReLU(6.0, name=f"relu_{index:02d}")(x)
        
    # 4. Classification Head
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling2d")(x)
    x = tf.keras.layers.Dropout(0.20, name="dropout")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="output_layer")(x)
    
    return tf.keras.Model(inputs, outputs, name="separable_cnn_64")

model = build_separable_cnn_64()
total_params = model.count_params()

print("=" * 60)
print(f"🎯 Model Name: {model.name}")
print(f"🔢 Total Parameters: {total_params:,} parameters")
print(f"✅ Pass Parameter Limit (< 100,000): {total_params < 100_000}")
print("=" * 60)
model.summary()


## 4. จัดเตรียม Data Augmentation และการจัดการ Class Imbalance
เนื่องจากมังคุดสุกเกิน (`overripe`) มีจำนวนภาพในธรรมชาติจำกัด เราจึงใช้เทคนิค:
1. **In-Memory Oversampling:** สุ่มตัวอย่างคลาสส่วนน้อยให้มีจำนวนเท่ากับคลาสส่วนใหญ่ใน Train Set
2. **Camera-Robust Augmentation:** จำลองความผันแปรของกล้อง OV2640 (การหมุน, ซูม, เลื่อน, ปรับความสว่าง, คอนทราสต์ และความอิ่มตัวสี) เพื่อให้โมเดลทนทานต่อสภาพแสงจริง


In [ ]:
SEED = 42
BATCH_SIZE = 16

# 1. In-Memory Oversampling เพื่อสร้างความสมดุลให้กับคลาสในการฝึกสอน
original_counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(int).tolist()
target_count = int(max(original_counts))
balanced_indices = []

for class_index in range(NUM_CLASSES):
    class_indices = np.flatnonzero(y_train == class_index)
    repeats = np.resize(class_indices, target_count)
    balanced_indices.append(repeats)

balanced_indices = np.concatenate(balanced_indices)
np.random.default_rng(SEED).shuffle(balanced_indices)

train_x_balanced = x_train[balanced_indices]
train_y_balanced = y_train[balanced_indices]

print(f"📊 Class distribution ก่อน Oversampling: {original_counts}")
print(f"⚖️ Class distribution หลัง Oversampling: {np.bincount(train_y_balanced, minlength=NUM_CLASSES).tolist()}")

# 2. Camera-Robust Augmentation Pipeline
augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    tf.keras.layers.RandomRotation(0.12, seed=SEED),
    tf.keras.layers.RandomZoom(0.20, seed=SEED),
    tf.keras.layers.RandomTranslation(0.12, 0.12, seed=SEED),
    tf.keras.layers.RandomContrast(0.35, seed=SEED),
    tf.keras.layers.RandomBrightness(0.20, value_range=(0, 255), seed=SEED),
]
data_augmentation = tf.keras.Sequential(augmentation_layers, name="camera_robust_augmentation")

def augment_batch(x, y):
    x = data_augmentation(x, training=True)
    x = tf.image.random_saturation(x, 0.55, 1.35, seed=SEED)
    x = tf.clip_by_value(x, 0.0, 255.0)
    return x, y

train_ds = tf.data.Dataset.from_tensor_slices((train_x_balanced, train_y_balanced)).shuffle(300, seed=SEED)
train_ds = train_ds.batch(BATCH_SIZE).map(augment_batch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE)

print("✅ tf.data.Dataset pipeline พร้อมสำหรับการฝึกสอน")


### แสดงตัวอย่างภาพหลังผ่าน Camera-Robust Augmentation


In [ ]:
# แสดงผลการทำ Augmentation จากภาพต้นฉบับ 1 ภาพ
sample_img = tf.expand_dims(x_train[0], 0)
plt.figure(figsize=(12, 3))
for i in range(4):
    aug_img, _ = augment_batch(sample_img, [0])
    plt.subplot(1, 4, i + 1)
    plt.imshow(aug_img[0].numpy().astype(np.uint8))
    plt.title(f"Augmented #{i+1}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 5. การฝึกสอนโมเดลแบบ 2 ขั้นตอน (Two-Stage Training)
1. **Stage 1 (Initial / Head Training):** ใช้ Learning Rate $10^{-3}$ เทรนโมเดลเริ่มต้น 20 epochs มี EarlyStopping เพื่อป้องกัน Overfitting
2. **Stage 2 (Fine-Tuning):** ลด Learning Rate ลงมาที่ $3.5 \times 10^{-5}$ พร้อม `ReduceLROnPlateau` เทรนต่อ 25 epochs เพื่อเก็บรายละเอียดและปรับน้ำหนักให้แม่นยำสูงสุด


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "output_colab_run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------
# Stage 1: Initial / Head Training
# ----------------------------------------------------
print("🚀 เริ่มต้น Stage 1: Initial Training (Learning Rate = 1e-3)...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

head_callbacks = [
    tf.keras.callbacks.CSVLogger(str(OUTPUT_DIR / "head_history.csv")),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=head_callbacks,
    verbose=1,
)

# ----------------------------------------------------
# Stage 2: Fine-Tuning
# ----------------------------------------------------
print("\n🎯 เริ่มต้น Stage 2: Fine-Tuning (Learning Rate = 3.5e-5)...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3.5e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

fine_callbacks = [
    tf.keras.callbacks.CSVLogger(str(OUTPUT_DIR / "fine_tune_history.csv")),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=fine_callbacks,
    verbose=1,
)

# บันทึกโมเดล Keras
keras_path = OUTPUT_DIR / "separable_cnn_64.keras"
model.save(keras_path)
print(f"\n💾 บันทึก Keras Model เรียบร้อยแล้ว: {keras_path}")


## 6. กราฟผลการฝึกสอนและประเมินผล Float32 Model (Evaluation)
ประเมินผลบนชุดทดสอบ (Test Set 38 ภาพ) พร้อมแสดง Confusion Matrix และค่า Classification Report


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# รวม Loss และ Accuracy ของทั้งสองช่วง
acc_head = history_head.history['accuracy']
val_acc_head = history_head.history['val_accuracy']
loss_head = history_head.history['loss']
val_loss_head = history_head.history['val_loss']

acc_fine = history_fine.history['accuracy']
val_acc_fine = history_fine.history['val_accuracy']
loss_fine = history_fine.history['loss']
val_loss_fine = history_fine.history['val_loss']

total_acc = acc_head + acc_fine
total_val_acc = val_acc_head + val_acc_fine
total_loss = loss_head + loss_fine
total_val_loss = val_loss_head + val_loss_fine

# วาดกราฟ Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(total_acc, label='Train Accuracy', color='#2b5c8f', lw=2)
ax1.plot(total_val_acc, label='Val Accuracy', color='#d95f02', lw=2)
ax1.axvline(x=len(acc_head)-1, color='gray', linestyle='--', label='Fine-tune Start')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

ax2.plot(total_loss, label='Train Loss', color='#2b5c8f', lw=2)
ax2.plot(total_val_loss, label='Val Loss', color='#d95f02', lw=2)
ax2.axvline(x=len(loss_head)-1, color='gray', linestyle='--', label='Fine-tune Start')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

# ประเมินผลบน Test Set ด้วย Float32 Model
float_preds = np.argmax(model.predict(x_test, verbose=0), axis=1)
float_acc = accuracy_score(y_test, float_preds)
float_bal_acc = balanced_accuracy_score(y_test, float_preds)

print("=" * 60)
print(f"🏆 ผลลัพธ์ Float32 บน Test Set:")
print(f"  - Accuracy         : {float_acc * 100:.2f}% ({np.sum(y_test == float_preds)}/{len(y_test)})")
print(f"  - Balanced Accuracy: {float_bal_acc * 100:.2f}%")
print("=" * 60)
print("\n📋 Classification Report (Float32):")
print(classification_report(y_test, float_preds, target_names=CLASSES, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, float_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"Float32 Confusion Matrix (Acc: {float_acc*100:.1f}%)")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_float32.png", dpi=150)
plt.show()


## 7. แปลงเป็น Full INT8 Quantization (TFLite for Microcontrollers)
บน **ESP32-S3 (TFLM)** จำเป็นต้องใช้ **Full INT8 Quantization** (ทั้ง Input, Output และ Weights เป็น `int8`) เพื่อประสิทธิภาพสูงสุดและการใช้ Hardware Acceleration (ESP-NN / Vector instructions)

เราใช้ **Representative Dataset Generator** เพื่อหาช่วงการบีบอัด Quantization Scale และ Zero-Point ที่แม่นยำที่สุด


In [ ]:
# เตรียมชุดตัวอย่างสำหรับ Calibrate INT8 Quantization (ใช้ Train + Val)
calibration_images = np.concatenate([x_train[:min(80, len(x_train))], x_val], axis=0)

def representative_data_gen():
    for img in calibration_images:
        # TFLite ต้องการ input dimension [1, 96, 96, 3] แบบ float32 ในการ calibrate
        yield [np.expand_dims(img, 0).astype(np.float32)]

# แปลงโมเดลด้วย TFLiteConverter
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

int8_model_bytes = converter.convert()

tflite_path = OUTPUT_DIR / "separable_cnn_64_int8.tflite"
tflite_path.write_bytes(int8_model_bytes)

file_bytes = len(int8_model_bytes)
file_kib = file_bytes / 1024
print("=" * 60)
print(f"📦 บันทึกโมเดล TFLite INT8 เรียบร้อยแล้ว:")
print(f"  - ไฟล์   : {tflite_path}")
print(f"  - ขนาด   : {file_bytes:,} bytes ({file_kib:.2f} KiB)")
print("=" * 60)

# ทดสอบรันโมเดล INT8 บน Test Set โดยตรงผ่าน TFLite Interpreter
interpreter = tf.lite.Interpreter(model_content=int8_model_bytes)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale, input_zero_point = input_details["quantization"]
output_scale, output_zero_point = output_details["quantization"]

print(f"⚙️ Input Quantization : Scale={input_scale:.6f}, ZeroPoint={input_zero_point}")
print(f"⚙️ Output Quantization: Scale={output_scale:.6f}, ZeroPoint={output_zero_point}")

int8_predictions = []
for img in x_test:
    # Quantize ภาพ float32 -> int8 ตาม scale & zero_point ของโมเดล
    quantized_img = np.clip(np.round(img / input_scale + input_zero_point), -128, 127).astype(np.int8)
    interpreter.set_tensor(input_details["index"], np.expand_dims(quantized_img, 0))
    interpreter.invoke()
    output_tensor = interpreter.get_tensor(output_details["index"])[0]
    int8_predictions.append(int(np.argmax(output_tensor)))

int8_predictions = np.asarray(int8_predictions)
int8_acc = accuracy_score(y_test, int8_predictions)
int8_bal_acc = balanced_accuracy_score(y_test, int8_predictions)

print("\n" + "=" * 60)
print(f"🏆 ผลลัพธ์ Full INT8 บน Test Set:")
print(f"  - Accuracy         : {int8_acc * 100:.2f}% ({np.sum(y_test == int8_predictions)}/{len(y_test)})")
print(f"  - Balanced Accuracy: {int8_bal_acc * 100:.2f}%")
print(f"  - Delta (INT8 - Float32 Accuracy Drop): {(int8_acc - float_acc)*100:+.2f}%")
print("=" * 60)
print("\n📋 Classification Report (Full INT8):")
print(classification_report(y_test, int8_predictions, target_names=CLASSES, digits=4))

# Confusion Matrix ของ INT8
cm_int8 = confusion_matrix(y_test, int8_predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_int8, annot=True, fmt='d', cmap='Greens', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"INT8 Confusion Matrix (Acc: {int8_acc*100:.1f}%)")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_int8.png", dpi=150)
plt.show()


## 8. แปลงโมเดลเป็น C/C++ Array สำหรับ ESP32-S3 (Firmware Deployment)
สำหรับ ESP32-S3 เราจะแปลงไฟล์ `.tflite` ให้อยู่ในรูป C source (`mangosteen_model_data.cc`) และ header (`mangosteen_model_data.h`) พร้อมกำหนด **`alignas(16)`** เพื่อประสิทธิภาพสูงสุดของ TensorFlow Lite Micro


In [ ]:
def convert_tflite_to_c_array(tflite_bytes: bytes, out_dir: Path, prefix="mangosteen_model"):
    out_dir.mkdir(parents=True, exist_ok=True)
    header_path = out_dir / f"{prefix}_data.h"
    source_path = out_dir / f"{prefix}_data.cc"
    
    model_len = len(tflite_bytes)
    array_name = f"g_{prefix}_data"
    len_name = f"g_{prefix}_data_len"
    
    # 1. เขียนไฟล์ Header (.h)
    header_guard = f"{prefix.upper()}_DATA_H_"
    header_content = f"""// Auto-generated from separable_cnn_64_int8.tflite
#ifndef {header_guard}
#define {header_guard}

#ifdef __cplusplus
extern "C" {{
#endif

extern const unsigned char {array_name}[];
extern const unsigned int {len_name};

#ifdef __cplusplus
}}
#endif

#endif  // {header_guard}
"""
    header_path.write_text(header_content, encoding="utf-8")
    
    # 2. เขียนไฟล์ Source (.cc) พร้อม alignas(16)
    with open(source_path, "w", encoding="utf-8") as f:
        f.write(f'// Auto-generated from separable_cnn_64_int8.tflite\n')
        f.write(f'#include "{prefix}_data.h"\n\n')
        f.write('// 16-byte alignment is required for TensorFlow Lite Micro tensor arena optimizations\n')
        f.write(f'alignas(16) const unsigned char {array_name}[] = {{\n')
        
        bytes_per_line = 12
        for i in range(0, model_len, bytes_per_line):
            chunk = tflite_bytes[i:i + bytes_per_line]
            hex_vals = [f"0x{b:02x}" for b in chunk]
            line = "  " + ", ".join(hex_vals)
            if i + bytes_per_line < model_len:
                line += ","
            f.write(line + "\n")
            
        f.write("};\n\n")
        f.write(f"const unsigned int {len_name} = {model_len};\n")
        
    print(f"✅ สร้างไฟล์ C Header สำเร็จ: {header_path}")
    print(f"✅ สร้างไฟล์ C Source สำเร็จ: {source_path} ({model_len:,} bytes)")
    return header_path, source_path

header_file, source_file = convert_tflite_to_c_array(int8_model_bytes, OUTPUT_DIR)


## 9. บันทึกรายงานสรุปและดาวน์โหลด Artifacts (Download for Deployment)
รันเซลล์นี้เพื่อบันทึก `summary.json` และรวมไฟล์โมเดลเป็น `.zip` พร้อมดาวน์โหลดลงคอมพิวเตอร์ของคุณเพื่อนำไปใส่ในโฟลเดอร์ `src/` ของโปรเจกต์ PlatformIO


In [ ]:
import json
import shutil

summary_data = {
    "model_name": "separable_cnn_64",
    "parameters": int(total_params),
    "parameter_limit": 100000,
    "input_shape": [1, 96, 96, 3],
    "classes": CLASSES,
    "float32_accuracy": float(float_acc),
    "float32_balanced_accuracy": float(float_bal_acc),
    "int8_accuracy": float(int8_acc),
    "int8_balanced_accuracy": float(int8_bal_acc),
    "int8_file_bytes": int(file_bytes),
    "int8_file_kib": float(file_kib),
    "quantization_input": {"scale": float(input_scale), "zero_point": int(input_zero_point)},
    "quantization_output": {"scale": float(output_scale), "zero_point": int(output_zero_point)},
}

summary_json_path = OUTPUT_DIR / "summary.json"
summary_json_path.write_text(json.dumps(summary_data, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"📄 บันทึกผลลัพธ์: {summary_json_path}")

# สร้างไฟล์ zip สำหรับดาวน์โหลด
zip_base = PROJECT_ROOT / "mangosteen_separable_cnn_64_artifacts"
zip_path = shutil.make_archive(str(zip_base), 'zip', str(OUTPUT_DIR))
print(f"📦 รวมไฟล์ทั้งหมดลง Zip: {zip_path}")

# ดาวน์โหลดผ่าน Google Colab files API (หากรันบน Colab)
try:
    from google.colab import files
    print("⬇️ กำลังส่งไฟล์ zip ให้ดาวน์โหลดลงเครื่อง...")
    files.download(zip_path)
except Exception:
    print(f"ℹ️ รันใน Local: สามารถเปิดไฟล์ผลลัพธ์ได้ที่ {zip_path} หรือโฟลเดอร์ {OUTPUT_DIR}")


## 10. วิธีนำไฟล์ไปใช้งานบนบอร์ดจริง (Deployment Guide)
หลังจากดาวน์โหลดไฟล์เรียบร้อยแล้ว ให้นำไปแฟลชลงบอร์ด LilyGO T-SIMCAM ดังนี้:

1. **แตกไฟล์ Zip** ที่ดาวน์โหลดมา
2. นำไฟล์ **`mangosteen_model_data.cc`** และ **`mangosteen_model_data.h`** ไปวางทับในโฟลเดอร์:
   ```text
   d:/MiniProject mangosteen/src/
   ```
3. นำไฟล์ **`separable_cnn_64_int8.tflite`** ไปเก็บไว้ใน:
   ```text
   d:/MiniProject mangosteen/current_model/model/separable_cnn_64_int8.tflite
   ```
4. เสียบสาย USB ระหว่างคอมพิวเตอร์กับบอร์ด LilyGO T-SIMCAM (ESP32-S3)
5. รันคำสั่งคอมไพล์และอัปโหลดเฟิร์มแวร์ผ่าน PlatformIO:
   ```powershell
   pio run -t upload
   ```
6. เปิด Serial Monitor เพื่อดูผลการตรวจจับมังคุดแบบเรียลไทม์:
   ```powershell
   pio device monitor
   ```
   หรือเปิดแดชบอร์ดกล้องผ่าน Web Browser ตาม IP Address ของ ESP32-S3 🌐
